In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))
import pandas as pd
import config 
import schema

In [2]:
facts = schema.resolution_facts(lat=18.0)
for k, v in facts.items():
    print(f"{k:35s} {v:,}" if isinstance(v, int) else f"{k:35s} {v}")

latitude                            18.0
cell_width_km                       10.59
cell_height_km                      11.13
cell_area_km2                       117.9
worldcover_pixels_per_side          1,200
worldcover_pixels_per_cell          1,440,000
viirs_pixels_per_cell               838
era5_values_per_cell_per_day        24


In [3]:
print("GRAIN          : one row = one grid cell on one day")
print(f"Grid size      : {schema.GRID_SIZE_DEG} degrees (ERA5-Land grid)")
print(f"Primary key    : {schema.PRIMARY_KEY}")
print(f"Total columns  : {len(schema.SCHEMA)}")
print()
schema.schema_table()

GRAIN          : one row = one grid cell on one day
Grid size      : 0.1 degrees (ERA5-Land grid)
Primary key    : ['grid_id', 'date']
Total columns  : 23



,column,dtype,unit,source,valid_range,description
0,grid_id,object,,derived,,"Grid cell identifier, e.g. '18.5_99.0'"
1,date,datetime64[ns],,derived,,The day this row describes
2,latitude,float64,degrees,derived,-90 .. 90,Latitude of the grid cell centre
3,longitude,float64,degrees,derived,-180 .. 180,Longitude of the grid cell centre
4,country,object,,derived,,"ISO-3 country code, e.g. 'THA'"
5,year,int64,,derived,2020 .. 2025,"Calendar year, for grouping"
6,month,int64,,derived,1 .. 12,"Calendar month, for seasonal analysis"
7,fire,int64,,FIRMS,0 .. 1,LABEL for the classification model (Session 18)
8,hotspot_count,int64,detections,FIRMS,0 .. None,"How many detections - a measure of extent, not..."
9,frp_max,float64,MW,FIRMS,0 .. None,Peak fire intensity in the cell that day


In [4]:
schema.schema_table().groupby("source")["column"].apply(list).to_frame()

,column
source,
ERA5-Land,"[temp_max_c, temp_mean_c, rainfall_mm, rainfal..."
FIRMS,"[fire, hotspot_count, frp_max, frp_sum]"
WorldCover,"[land_cover_dominant, forest_pct, cropland_pct..."
derived,"[grid_id, date, latitude, longitude, country, ..."


In [5]:
for lat, lon in [(18.52, 99.03), (18.58, 99.08), (17.91, 98.44)]:
    gid = schema.latlon_to_grid_id(lat, lon)
    back = schema.grid_id_to_latlon(gid)
    print(f"({lat}, {lon})  ->  {gid:>14}  -> cell centre: {back}")

(18.52, 99.03)  ->       18.5_99.0  -> cell centre: (18.5, 99.0)
(18.58, 99.08)  ->       18.6_99.1  -> cell centre: (18.6, 99.1)
(17.91, 98.44)  ->       17.9_98.4  -> cell centre: (17.9, 98.4)


In [6]:
hotspots = [(18.52, 99.03), (18.54, 99.01), (18.48, 98.97), (18.91, 99.42)]

for lat, lon in hotspots:
    print(f"({lat}, {lon}) -> {schema.latlon_to_grid_id(lat, lon)}")

print()
print("Hotspots  :", len(hotspots))
print("Grid cells:", len({schema.latlon_to_grid_id(la, lo) for la, lo in hotspots}))

(18.52, 99.03) -> 18.5_99.0
(18.54, 99.01) -> 18.5_99.0
(18.48, 98.97) -> 18.5_99.0
(18.91, 99.42) -> 18.9_99.4

Hotspots  : 4
Grid cells: 2


In [7]:
schema.empty_dataframe().dtypes

grid_id                        object
date                   datetime64[ns]
latitude                      float64
longitude                     float64
country                        object
year                            int64
month                           int64
fire                            int64
hotspot_count                   int64
frp_max                       float64
frp_sum                       float64
temp_max_c                    float64
temp_mean_c                   float64
rainfall_mm                   float64
rainfall_7d_mm                float64
dry_days                        int64
wind_speed_kmh                float64
soil_moisture                 float64
land_cover_dominant            object
forest_pct                    float64
cropland_pct                  float64
grassland_pct                 float64
other_pct                     float64
dtype: object

In [8]:
def make_row(grid_id, date, fire, **kwargs):
    lat, lon = schema.grid_id_to_latlon(grid_id)
    d = pd.Timestamp(date)
    row = dict(
        grid_id=grid_id, date=d, latitude=lat, longitude=lon, country="THA",
        year=d.year, month=d.month,
        fire=fire,
        hotspot_count=0, frp_max=0.0, frp_sum=0.0,
        temp_max_c=32.0, temp_mean_c=25.0,
        rainfall_mm=0.0, rainfall_7d_mm=0.0, dry_days=0,
        wind_speed_kmh=8.0, soil_moisture=0.25,
        land_cover_dominant="forest",
        forest_pct=60.0, cropland_pct=20.0, grassland_pct=15.0, other_pct=5.0,
    )
    row.update(kwargs)
    return row

sample_rows = [
    make_row("18.5_99.0", "2025-03-15", fire=1, hotspot_count=7, frp_max=11.5,
             frp_sum=32.4, temp_max_c=36.2, rainfall_mm=0.0, rainfall_7d_mm=0.8,
             dry_days=14, soil_moisture=0.12, land_cover_dominant="cropland",
             forest_pct=25.0, cropland_pct=58.0, grassland_pct=12.0, other_pct=5.0),

    make_row("18.5_99.0", "2025-03-16", fire=1, hotspot_count=3, frp_max=6.1,
             frp_sum=11.2, temp_max_c=35.8, rainfall_mm=0.0, rainfall_7d_mm=0.8,
             dry_days=15, soil_moisture=0.11, land_cover_dominant="cropland",
             forest_pct=25.0, cropland_pct=58.0, grassland_pct=12.0, other_pct=5.0),

    make_row("18.6_99.1", "2025-03-15", fire=0, temp_max_c=31.4, rainfall_mm=12.5,
             rainfall_7d_mm=48.2, dry_days=0, soil_moisture=0.34,
             forest_pct=78.0, cropland_pct=8.0, grassland_pct=10.0, other_pct=4.0),
]

sample = pd.DataFrame(sample_rows)

for col, spec in schema.SCHEMA.items():
    if spec["dtype"].startswith("int"):
        sample[col] = sample[col].astype("int64")
    elif spec["dtype"].startswith("float"):
        sample[col] = sample[col].astype("float64")

sample

,grid_id,date,latitude,longitude,country,year,month,fire,hotspot_count,frp_max,...,rainfall_mm,rainfall_7d_mm,dry_days,wind_speed_kmh,soil_moisture,land_cover_dominant,forest_pct,cropland_pct,grassland_pct,other_pct
0,18.5_99.0,2025-03-15,18.5,99.0,THA,2025,3,1,7,11.5,...,0.0,0.8,14,8.0,0.12,cropland,25.0,58.0,12.0,5.0
1,18.5_99.0,2025-03-16,18.5,99.0,THA,2025,3,1,3,6.1,...,0.0,0.8,15,8.0,0.11,cropland,25.0,58.0,12.0,5.0
2,18.6_99.1,2025-03-15,18.6,99.1,THA,2025,3,0,0,0.0,...,12.5,48.2,0,8.0,0.34,forest,78.0,8.0,10.0,4.0


In [9]:
problems = schema.validate_schema(sample)

validate_schema: OK - 3 rows, 23 columns


In [10]:
broken = sample.copy()
broken.loc[0, "forest_pct"] = 150.0      # > 100
broken.loc[1, "rainfall_mm"] = -5.0      # am
broken.loc[2, "soil_moisture"] = 1.8     # > 1

schema.validate_schema(broken)

validate_schema: 4 problem(s) found
  - Column 'rainfall_mm': 1 value(s) below minimum 0
  - Column 'soil_moisture': 1 value(s) above maximum 1
  - Column 'forest_pct': 1 value(s) above maximum 100
  - 1 row(s) where the four land-cover percentages do not sum to 100


["Column 'rainfall_mm': 1 value(s) below minimum 0",
 "Column 'soil_moisture': 1 value(s) above maximum 1",
 "Column 'forest_pct': 1 value(s) above maximum 100",
 '1 row(s) where the four land-cover percentages do not sum to 100']

In [11]:
schema.validate_schema(sample.drop(columns=["rainfall_7d_mm", "dry_days"]))

validate_schema: 1 problem(s) found
  - Missing columns: ['rainfall_7d_mm', 'dry_days']


["Missing columns: ['rainfall_7d_mm', 'dry_days']"]

In [12]:
dupes = schema.check_primary_key(sample)

check_primary_key: OK - all 3 rows have a unique (grid_id, date)


In [13]:
with_dupes = pd.concat([sample, sample.iloc[[0]]], ignore_index=True)
schema.check_primary_key(with_dupes)

check_primary_key: FAILED - 2 row(s) share a (grid_id, date) with another row
  grid_id       date
18.5_99.0 2025-03-15
18.5_99.0 2025-03-15


,grid_id,date,latitude,longitude,country,year,month,fire,hotspot_count,frp_max,...,rainfall_mm,rainfall_7d_mm,dry_days,wind_speed_kmh,soil_moisture,land_cover_dominant,forest_pct,cropland_pct,grassland_pct,other_pct
0,18.5_99.0,2025-03-15,18.5,99.0,THA,2025,3,1,7,11.5,...,0.0,0.8,14,8.0,0.12,cropland,25.0,58.0,12.0,5.0
3,18.5_99.0,2025-03-15,18.5,99.0,THA,2025,3,1,7,11.5,...,0.0,0.8,14,8.0,0.12,cropland,25.0,58.0,12.0,5.0


In [14]:
west, south, east, north = (float(x) for x in config.SEA_BBOX.split(","))

n_cells = ((east - west) / schema.GRID_SIZE_DEG) * ((north - south) / schema.GRID_SIZE_DEG)
n_days = (pd.Timestamp(config.STUDY_END) - pd.Timestamp(config.STUDY_START)).days + 1

print(f"Bounding box (SEA) : {config.SEA_BBOX}")
print(f"Grid cells         : {n_cells:,.0f}")
print(f"Days 2020-2025     : {n_days:,}")
print(f"Full cross product : {n_cells * n_days:,.0f} rows")
print()
print(f"Estimated size (~200 bytes/row): {n_cells * n_days * 200 / 1e9:,.0f} GB")

Bounding box (SEA) : 92,-11,141,29
Grid cells         : 196,000
Days 2020-2025     : 2,192
Full cross product : 429,632,000 rows

Estimated size (~200 bytes/row): 86 GB
